# Batching

In machine learning applications, we need to divide the training data into smaller chunks of data to feed into the model. For instance, a dataset that has 10,000 samples might be divided into batches of 100 samples, leading to 100 batches per training epoch.

To do this with the data loader, we use the `crest.src.data.Batcher` class. By passing in a Dataset object, we can start creating batches of data immediately rather than loading the entire dataset at once. Similarly, because Dataset uses dask as the backend framework, we can pull different parts of the data to create multiple batches in parallel. 

In [1]:
# Ensure we can import crest
%cd -q ..

To demonstrate the Batcher itself however, we're going to look at the MNIST dataset in order to minimize the complexity.  

In [2]:
# Load MNIST
import tensorflow as tf
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

Since this data can fit in memory, and features/samples are already aligned, we can use a more specialized version of Dataset: `crest.src.data.loading.StructuredDataset`. 

`StructuredDataset` requires that all features align on their first dimension, which is assumed to be the number of samples. This is typical for most datasets that are able to be passed as-is to a machine learning model - and MNIST is no exception. 

Using `StructuredDataset` allows us to skip all of the heterogeneous grid alignment and out-of-memory data handling logic, which significantly speeds up the rate at which we can generate batches. For completeness however, the use of the more generally applicable `Dataset` class will be demonstrated at the end of this notebook as well. 

In [3]:
from crest.data.loading.StructuredDataset import StructuredDataset

# To use StructuredDataset, we simply pass in each feature array
# As well, we can set the labels parameter to use specific labels for each array
ds_train = StructuredDataset(x_train, y_train, labels=['image', 'class'])
ds_test  = StructuredDataset(x_test,  y_test,  labels=['image', 'class'])

Now we can create our batchers, which will allow us to iterate over batches and/or train a machine learning model:

In [4]:
from crest.data.Batcher import Batcher

kwargs = {
    # We'll have 200 samples in each batch
    'batch_size' : 200,
    
    # Each batch should consist of two dictionaries: one for inputs, and one for outputs.
    # In this case we only have two features - image and class - which are the inputs 
    #   and outputs, respectively
    'features'   : [['image'], ['class']], 
    
    # We should yield batches indefinitely, repeating after reaching the end of an epoch
    'repeat'     : True, 
}

# For the trainer, there's one additional parameter we'll set: duplicate=True
# Setting duplicate to True means we can see the same samples multiple times in an epoch
# This is required because we're using StructuredDataset, which by default puts all of
#   the data into a single block in the dask array. 

# If we want to use multiple workers to generate samples however, there need to either be:
#   - multiple blocks (which we could change in the StructuredDataset object, but it would be less efficient)
#   - duplicate samples allowed (so that each worker can operate on the same block)
batch_train = Batcher(ds_train, duplicate=True, **kwargs)

# For the test batcher speed is less of a concern, and we don't want to bias accuracy metrics by
# having duplicate samples anyway. So, we'll just set the number of workers to be 0 in this case 
# (which means batches will be generated in the main thread rather than a background process). 
# Note that the default number of workers is 2. 
batch_test  = Batcher(ds_test,  workers=0, **kwargs)

Before we actually train a model, let's check the performance of just iterating over batches. Iterating batches can be done by just looping over the batcher itself (e.g. `for batch in batch_train: ...`).

However, to show performance metrics, we can also use Batcher.generator(show_timing=True). This should be able to iterate at  ~1.25k Batches/s, or ~250k Samples/s:

In [5]:
for i, batch in enumerate( batch_train.generator(show_timing=True) ):
    if i > 20000: break

0.00 Batches [00:00, ? Batches/s]

0.00 Samples [00:00, ? Samples/s]

Time to first batch: 35.8 seconds



There are two things to note:
- the time to first batch is mostly impacted by the number of workers we request, since there's overhead in spawning new processes
- once a Batcher begins generating batches, it will continue to do so in the background until Batcher.close() is called on it (up to the max queue size, which is 100 batches by default)

To demonstrate the first point, we can look at time to first batch on batch_test, for which we had set workers=0:

In [6]:
for i, batch in enumerate( batch_test.generator(show_timing=True) ):
    if i > 1: break

0.00 Batches [00:00, ? Batches/s]

0.00 Samples [00:00, ? Samples/s]

Time to first batch: 0.7 seconds



As for the second point - that batches will continue to be generated in the background - it means that if we try iterating batch_train again, the time to first batch will be 0 since there are batches waiting in the queue to be consumed:

In [7]:
for i, batch in enumerate( batch_train.generator(show_timing=True) ):
    if i > 20000: break

0.00 Batches [00:00, ? Batches/s]

0.00 Samples [00:00, ? Samples/s]

Time to first batch: 0.0 seconds



As a final note before we train a model, Batcher can be used in a context manager to automatically call Batcher.close() once the block is exited. This ensures all resources are released once you're finished with them, but also means background processes will need to be recreated if you end up wanting to generate batches again (though this will take place automatically when you try iterating over the batcher): 

In [8]:
print('Active workers before closing:', batch_train.active_workers)
print()

# The same could be done by manually calling batch_train.close()
with batch_train as batches:
    for (x, y) in batches:
        print('Input: ', {k: v.shape for k,v in x.items()})
        print('Target:', {k: v.shape for k,v in y.items()})
        break
        
print('Active workers after closing:', batch_train.active_workers)
print()

# Fetching another batch will restart the background processes
x, y = next(batch_train)
print('Input: ', {k: v.shape for k,v in x.items()})
print('Target:', {k: v.shape for k,v in y.items()})
print('Active workers after restarting:', batch_train.active_workers)

Active workers before closing: [<SpawnProcess name='SpawnProcess-1' pid=75038 parent=74939 started daemon>, <SpawnProcess name='SpawnProcess-2' pid=75111 parent=74939 started daemon>]

Input:  {'image': (200, 28, 28)}
Target: {'class': (200,)}
Active workers after closing: []



Input:  {'image': (200, 28, 28)}
Target: {'class': (200,)}
Active workers after restarting: [<SpawnProcess name='SpawnProcess-3' pid=75593 parent=74939 started daemon>, <SpawnProcess name='SpawnProcess-4' pid=75668 parent=74939 started daemon>]


## Training a model

The Batcher is designed to work seamlessly with tensorflow (and any other framework that can take a generator of dictionaries). Let's build a simple model to demonstrate:

In [9]:
from tensorflow.keras.regularizers import L2
from tensorflow.keras.layers import Conv2D, Dropout, Flatten, Dense
from tensorflow.keras import Input, Model

n_classes  = 10
n_features = 1
patch_size = x_train.shape[1]
x = inputs = Input(shape=(patch_size, patch_size, n_features))
x = Conv2D(filters=10, kernel_size=3, padding='same', activation='relu', kernel_regularizer=L2(1e-4))(x)
x = Dropout(0.2)(x)
x = Flatten()(x)
x = output = tf.keras.layers.Dense(n_classes, activation='softmax', kernel_regularizer=L2(1e-3))(x)

# We only need to specify which features in the dictionaries go to which objects
model = Model({'image':inputs}, {'class':output}, name='SimpleCNN')
model.summary()

Model: "SimpleCNN"


_________________________________________________________________


 Layer (type)                Output Shape              Param #   


 input_1 (InputLayer)        [(None, 28, 28, 1)]       0         


 conv2d (Conv2D)             (None, 28, 28, 10)        100       


...

 dense (Dense)               (None, 10)                78410     


Total params: 78,510


Trainable params: 78,510


Non-trainable params: 0


_________________________________________________________________


In [10]:
fit_kwargs = {
    'steps_per_epoch'  : 300, # 60k samples / 200 samples per batch
    'validation_steps' : 50,  # 10k samples / 200 samples per batch
    'epochs'           : 5,
    'verbose'          : 1,
}

#### Using this model, we should be able to get > 97% test accuracy in about 30 seconds:

In [11]:
%%time
model = tf.keras.models.clone_model(model) # Reset model weights
model.compile('Adam', loss='sparse_categorical_crossentropy', metrics=['sparse_categorical_accuracy'])
model.fit(batch_train, validation_data=batch_test, **fit_kwargs)

Epoch 1/5


  1/300 [..............................] - ETA: 1:48 - loss: 67.1337 - sparse_categorical_accuracy: 0.0950

  4/300 [..............................] - ETA: 5s - loss: 45.2779 - sparse_categorical_accuracy: 0.2600  

  7/300 [..............................] - ETA: 5s - loss: 33.8226 - sparse_categorical_accuracy: 0.3779

 10/300 [>.............................] - ETA: 5s - loss: 26.7540 - sparse_categorical_accuracy: 0.4795

...

294/300 [============================>.] - ETA: 0s - loss: 0.1226 - sparse_categorical_accuracy: 0.9743

297/300 [============================>.] - ETA: 0s - loss: 0.1218 - sparse_categorical_accuracy: 0.9744

300/300 [==============================] - ETA: 0s - loss: 0.1215 - sparse_categorical_accuracy: 0.9746

300/300 [==============================] - 6s 21ms/step - loss: 0.1215 - sparse_categorical_accuracy: 0.9746 - val_loss: 0.1478 - val_sparse_categorical_accuracy: 0.9730


CPU times: user 2min 42s, sys: 9.2 s, total: 2min 51s
Wall time: 30.9 s


The bottleneck at this point is the model itself, as tensorflow can't feed batches through any quicker. This can be seen by simply passing in the data itself to the model, which trains in a similar time (minus a small overhead that tensorflow creates when using a generator):

In [12]:
%%time
model = tf.keras.models.clone_model(model) # Reset model weights
model.compile('Adam', loss='sparse_categorical_crossentropy', metrics=['sparse_categorical_accuracy'])
model.fit(x_train, y_train, validation_data=(x_test, y_test), **fit_kwargs)

Epoch 1/5


  1/300 [..............................] - ETA: 1:39 - loss: 47.2662 - sparse_categorical_accuracy: 0.0850

  5/300 [..............................] - ETA: 4s - loss: 29.7141 - sparse_categorical_accuracy: 0.2390  

  9/300 [..............................] - ETA: 4s - loss: 21.5579 - sparse_categorical_accuracy: 0.3811

 13/300 [>.............................] - ETA: 4s - loss: 17.2364 - sparse_categorical_accuracy: 0.4612

...

290/300 [============================>.] - ETA: 0s - loss: 0.0979 - sparse_categorical_accuracy: 0.9797

294/300 [============================>.] - ETA: 0s - loss: 0.0986 - sparse_categorical_accuracy: 0.9796

298/300 [============================>.] - ETA: 0s - loss: 0.0994 - sparse_categorical_accuracy: 0.9794

300/300 [==============================] - 5s 16ms/step - loss: 0.0994 - sparse_categorical_accuracy: 0.9794 - val_loss: 0.1599 - val_sparse_categorical_accuracy: 0.9739


CPU times: user 2min 32s, sys: 7.79 s, total: 2min 40s
Wall time: 24.8 s
